# Birkin vs Not-Birkin CNN (NumPy Layers)

This version mirrors the assignment flow and keeps only the two sections you requested.


## Initialize the model


In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import sys

import numpy as np
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Ensure imports work whether the notebook is opened from the repo root or the layers/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "layers":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from layers import Conv2D, MaxPool2D, Flatten, Linear, ReLU, Softmax, Sequential, CrossEntropyLoss

# Base configuration
BIRKIN_DIRS = [Path("Data/Birkin"), Path("Data/birkins")]
OTHER_DIR = Path("Data/other")
IMAGE_SIZE = 16
TEST_SIZE = 0.2
SEED = 42
MAX_SAMPLES_PER_CLASS = 120
REPORT_PATH = Path("outputs/pipeline_report.txt")

IMAGE_PATTERNS = ("*.jpg", "*.jpeg", "*.png")


def list_image_files(folder: Path) -> list[Path]:
    files = []
    for pattern in IMAGE_PATTERNS:
        files.extend(sorted(folder.glob(pattern)))
    return files


def load_image(path: Path, size: int) -> np.ndarray:
    image = Image.open(path).convert("L").resize((size, size), Image.Resampling.LANCZOS)
    return np.asarray(image, dtype=np.float32) / 255.0


def build_dataset(size: int, max_samples_per_class: int | None = None):
    birkin = []
    for folder in BIRKIN_DIRS:
        if folder.exists():
            birkin.extend(list_image_files(folder))
    other = list_image_files(OTHER_DIR)

    if not birkin or not other:
        raise ValueError("Could not find images in Data/Birkin|Data/birkins and Data/other.")

    if max_samples_per_class is not None:
        birkin = birkin[:max_samples_per_class]
        other = other[:max_samples_per_class]

    x_birkin = [load_image(fp, size) for fp in birkin]
    x_other = [load_image(fp, size) for fp in other]

    x = np.stack(x_birkin + x_other, axis=0)
    y = np.array([1] * len(x_birkin) + [0] * len(x_other), dtype=np.int64)

    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(y))
    x, y = x[perm], y[perm]

    x = x[:, None, :, :]
    return x, y


def make_model(image_size: int, conv_channels: int):
    pooled = image_size // 2
    hidden = 32
    return Sequential([
        Conv2D(in_channels=1, out_channels=conv_channels, kernel_size=3, stride=1, padding=1),
        ReLU(),
        MaxPool2D(kernel_size=2, stride=2),
        Flatten(),
        Linear(conv_channels * pooled * pooled, hidden),
        ReLU(),
        Linear(hidden, 2),
        Softmax(),
    ])


def iter_batches(x, y, batch_size, rng):
    idx = np.arange(len(y))
    rng.shuffle(idx)
    for i in range(0, len(idx), batch_size):
        b = idx[i:i + batch_size]
        yield x[b], y[b]


def train_once(x_train, y_train, x_val, y_val, lr, batch_size, epochs, conv_channels, seed=42):
    rng = np.random.default_rng(seed)
    model = make_model(IMAGE_SIZE, conv_channels)
    loss_fn = CrossEntropyLoss()
    history = {"loss": [], "val_acc": []}

    for epoch in range(epochs):
        running_loss = []
        for xb, yb in iter_batches(x_train, y_train, batch_size, rng):
            probs = model.forward(xb)
            loss = loss_fn.forward(probs, yb)
            grad = loss_fn.backward()
            model.backward(grad)
            model.update(lr)
            model.zero_grad()
            running_loss.append(loss)

        val_pred = model.predict(x_val)
        val_acc = float((val_pred == y_val).mean())
        history["loss"].append(float(np.mean(running_loss)))
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch + 1}/{epochs} | loss={history['loss'][-1]:.4f} | val_acc={val_acc:.4f}")

    return model, history


# Tune your hyperparameters


In [ ]:
# Edit only this search space when tuning hyperparameters.
LEARNING_RATES = [0.01, 0.005]
BATCH_SIZES = [8, 16]
EPOCHS = [3]
CONV_CHANNELS = [4, 8]

x, y = build_dataset(size=IMAGE_SIZE, max_samples_per_class=MAX_SAMPLES_PER_CLASS)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

x_train_sub, x_val, y_train_sub, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

best = None
search_results = []

for lr, bs, ep, ch in product(LEARNING_RATES, BATCH_SIZES, EPOCHS, CONV_CHANNELS):
    print("\n=== trial ===")
    print({"lr": lr, "batch_size": bs, "epochs": ep, "conv_channels": ch})

    model, history = train_once(
        x_train_sub,
        y_train_sub,
        x_val,
        y_val,
        lr=lr,
        batch_size=bs,
        epochs=ep,
        conv_channels=ch,
        seed=SEED,
    )

    score = history["val_acc"][-1]
    result = {"lr": lr, "batch_size": bs, "epochs": ep, "conv_channels": ch, "val_acc": score}
    search_results.append(result)

    if best is None or score > best["val_acc"]:
        best = result

print("\nBest hyperparameters:", best)

best_model, _ = train_once(
    x_train,
    y_train,
    x_test,
    y_test,
    best["lr"],
    best["batch_size"],
    best["epochs"],
    best["conv_channels"],
    seed=SEED,
)

y_pred = best_model.predict(x_test)
acc = float((y_pred == y_test).mean())
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=["not_birkin", "birkin"], digits=4)

print(f"Test accuracy: {acc:.4f}")
print(cm)
print(report)

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
with REPORT_PATH.open("a", encoding="utf-8") as f:
    f.write("\n\n================ CNN (NumPy layers) ================\n")
    f.write(f"image_size={IMAGE_SIZE}x{IMAGE_SIZE}\n")
    f.write(f"train_samples={len(x_train)}\n")
    f.write(f"test_samples={len(x_test)}\n")
    f.write(f"max_samples_per_class={MAX_SAMPLES_PER_CLASS}\n")
    f.write("best_hyperparameters=\n")
    f.write(f"  learning_rate={best['lr']}\n")
    f.write(f"  batch_size={best['batch_size']}\n")
    f.write(f"  epochs={best['epochs']}\n")
    f.write(f"  conv_channels={best['conv_channels']}\n")
    f.write(f"test_accuracy={acc:.4f}\n")
    f.write("confusion_matrix=\n")
    f.write(str(cm) + "\n")
    f.write("classification_report=\n")
    f.write(report + "\n")

print(f"Appended CNN results to {REPORT_PATH}")
